In [1]:

import os, torch
import pandas as pd
import numpy as np
# from pathlib import Path
from pathlib import PureWindowsPath


import sys
sys.path.append("/projects/net_contrast_classification/contrast_phase")

from Radiomics.radiomics_pipeline import multi_channel
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold


In [2]:
def contrast_timing(dataset):
    dataset["contrast_timing"] = dataset["contrast"]
    mask = dataset["contrast"] != "Non-contrast"
    dataset.loc[mask, "contrast_timing"] = (dataset.loc[mask, "contrast"] + " " + dataset.loc[mask, "phase_timing"].astype(str))
    dataset.loc[mask, "contrast_timing"] = dataset.loc[mask, "contrast_timing"].str.strip()
    dataset.loc[dataset["contrast"] == "Non-contrast", "contrast_timing"] = "Non-contrast"   

    return dataset

def files_load(data_dir, sample=None):
    dataset = pd.read_csv(data_dir)

    dataset['server_folder'] = dataset.exist_on_server.apply(
        lambda x: "/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTINET"
        if pd.notna(x)
        else "/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTINET/not_on_server"
    )

    dataset = dataset[
        (dataset.contrast.isin(['Arterial', "Portal", "Non-contrast"])) &
        (dataset.is_liver_imaged.isin(["Yes", "Partially"])) &
        (dataset.phase_timing != '0.0')
    ]

    dataset.loc[dataset["contrast"] == "Non-contrast", "phase_timing"] = None
    dataset = contrast_timing(dataset)

    if sample is not None:
        dataset = (dataset.groupby('contrast_timing', dropna=False, group_keys=False)
                   .apply(lambda x: x.sample(n=min(len(x), sample), random_state=42))
                   .reset_index(drop=True))

        dataset = contrast_timing(dataset)
        dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

    # ---------------- BUILD FILE PATHS ----------------
    dataset["image_file"] = dataset.apply(
        lambda row: os.path.join(
            row["server_folder"],
            PureWindowsPath(row["MatchKey"]).name
        ),
        axis=1
    )

    dataset["organ_file"] = dataset["image_file"].str.replace(".nii.gz", ".organs.nii.gz")

    # ---------------- SELECT FINAL COLUMNS ----------------
    keep_cols = [
        "SubjectKeyRadiology",
        "ExamDate",
        "MatchKey",
        "contrast",
        "contrast_timing",
        "image_file",
        "organ_file",
    ]

    dataset = dataset[keep_cols].copy()

    dataset = dataset.rename(columns={
        "contrast_timing": "phase"
    })

    rare_classes = ["Portal Too Late",
                    "Portal Too Early",
                    "Arterial Too Late"]

    dataset["rare_class"] = dataset["phase"].isin(rare_classes).astype(int)

    return dataset

In [3]:
data_dir = "/projects/net_contrast_classification/contrast_phase/data/cleaned_data_1.csv"
dataset = files_load(data_dir, sample=None)
dataset

,SubjectKeyRadiology,ExamDate,MatchKey,contrast,phase,image_file,organ_file,rare_class
0,NKI-d23231-00-0063,2012-04-16,NKI-d23231-00-0063_20120416_kalina01_0000.nii.gz,Arterial,Arterial Too Early,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0
1,NKI-d23231-00-0070,2015-08-21,NKI-d23231-00-0070_20150821_kalina01_0001.nii.gz,Non-contrast,Non-contrast,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0
2,NKI-d23231-00-0036,2014-02-18,NKI-d23231-00-0036_20140218_kalina01_0002.nii.gz,Portal,Portal Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0
3,NKI-d23231-00-0071,2013-09-24,NKI-d23231-00-0071_20130924_kalina01_0003.nii.gz,Arterial,Arterial Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0
4,NKI-d23231-00-0077,2014-12-22,NKI-d23231-00-0077_20141222_kalina01_0004.nii.gz,Portal,Portal Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0
...,...,...,...,...,...,...,...,...
8729,NKI-d23231-00-0888,2022-04-20,NKI-d23231-00-0888_20220420_kalina99_5827.nii.gz,Portal,Portal Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0
8730,NKI-d23231-00-0888,2023-04-24,NKI-d23231-00-0888_20230424_kalina99_5828.nii.gz,Arterial,Arterial Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0
8731,NKI-d23231-00-0888,2023-04-24,NKI-d23231-00-0888_20230424_kalina99_5829.nii.gz,Portal,Portal Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0
8732,NKI-d23231-00-0888,2023-08-01,NKI-d23231-00-0888_20230801_kalina99_5830.nii.gz,Arterial,Arterial Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0


In [4]:

def group_stratified_train_val_test_split(indices, labels, groups,
                                         test_size=0.2, val_size=0.2,
                                         random_state=42):

    indices = np.array(indices)
    labels = np.array(labels)
    groups = np.array(groups)

    # -------------------------
    # STEP 1: train vs test
    # -------------------------
    n_splits_test = int(1 / test_size)

    sgkf_test = StratifiedGroupKFold(
        n_splits=n_splits_test,
        shuffle=True,
        random_state=random_state
    )

    train_val_idx, test_idx = next(
        sgkf_test.split(indices, y=labels, groups=groups)
    )

    # subset for train/val
    indices_tv = indices[train_val_idx]
    labels_tv = labels[train_val_idx]
    groups_tv = groups[train_val_idx]

    # -------------------------
    # STEP 2: train vs val
    # -------------------------

    val_relative = val_size / (1 - test_size)
    n_splits_val = int(1 / val_relative)

    sgkf_val = StratifiedGroupKFold(
        n_splits=n_splits_val,
        shuffle=True,
        random_state=random_state
    )

    train_idx_rel, val_idx_rel = next(
        sgkf_val.split(indices_tv, y=labels_tv, groups=groups_tv)
    )

    train_idx = indices_tv[train_idx_rel]
    val_idx = indices_tv[val_idx_rel]

    return train_idx, val_idx, test_idx


dataset = dataset.reset_index(drop=True)

train_idx, val_idx, test_idx = group_stratified_train_val_test_split(
    np.arange(len(dataset)),
    labels=dataset["phase"],
    groups=dataset["SubjectKeyRadiology"]
)

dataset["split"] = "unassigned"

dataset.loc[train_idx, "split"] = "train"
dataset.loc[val_idx, "split"] = "val"
dataset.loc[test_idx, "split"] = "test"


In [5]:
dataset.split.value_counts()

split
train    4883
val      1628
test     1628
Name: count, dtype: int64

In [18]:
print(dataset[dataset.split == "train"]["phase"].value_counts())
print("\n")
print(dataset[dataset.split == "test"]["phase"].value_counts())
print("\n")
print(dataset[dataset.split == "val"]["phase"].value_counts())



phase
Portal Just Right      2117
Non-contrast           1175
Arterial Just Right     940
Arterial Too Early      477
Portal Too Early         70
Arterial Too Late        69
Portal Too Late          35
Name: count, dtype: int64


phase
Portal Just Right      705
Non-contrast           391
Arterial Just Right    314
Arterial Too Early     160
Portal Too Early        23
Arterial Too Late       23
Portal Too Late         12
Name: count, dtype: int64


phase
Portal Just Right      706
Non-contrast           392
Arterial Just Right    313
Arterial Too Early     159
Portal Too Early        24
Arterial Too Late       23
Portal Too Late         11
Name: count, dtype: int64


In [ ]:
dataset["augment"] = ((dataset["split"] == "train") & (dataset["rare_class"] == 1)).astype(int)
dataset

,SubjectKeyRadiology,ExamDate,MatchKey,contrast,phase,image_file,organ_file,rare_class,split,augment
0,NKI-d23231-00-0063,2012-04-16,NKI-d23231-00-0063_20120416_kalina01_0000.nii.gz,Arterial,Arterial Too Early,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,val,0
1,NKI-d23231-00-0070,2015-08-21,NKI-d23231-00-0070_20150821_kalina01_0001.nii.gz,Non-contrast,Non-contrast,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,train,0
2,NKI-d23231-00-0036,2014-02-18,NKI-d23231-00-0036_20140218_kalina01_0002.nii.gz,Portal,Portal Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,test,0
3,NKI-d23231-00-0071,2013-09-24,NKI-d23231-00-0071_20130924_kalina01_0003.nii.gz,Arterial,Arterial Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,train,0
4,NKI-d23231-00-0077,2014-12-22,NKI-d23231-00-0077_20141222_kalina01_0004.nii.gz,Portal,Portal Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,train,0
...,...,...,...,...,...,...,...,...,...,...
8134,NKI-d23231-00-0888,2022-04-20,NKI-d23231-00-0888_20220420_kalina99_5827.nii.gz,Portal,Portal Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,train,0
8135,NKI-d23231-00-0888,2023-04-24,NKI-d23231-00-0888_20230424_kalina99_5828.nii.gz,Arterial,Arterial Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,train,0
8136,NKI-d23231-00-0888,2023-04-24,NKI-d23231-00-0888_20230424_kalina99_5829.nii.gz,Portal,Portal Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,train,0
8137,NKI-d23231-00-0888,2023-08-01,NKI-d23231-00-0888_20230801_kalina99_5830.nii.gz,Arterial,Arterial Just Right,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,0,train,0


In [25]:
dataset[dataset.augment == 1]

,SubjectKeyRadiology,ExamDate,MatchKey,contrast,phase,image_file,organ_file,rare_class,split,augment
43,NKI-d23231-00-0009,2017-01-18,NKI-d23231-00-0009_20170118_kalina01_0047.nii.gz,Portal,Portal Too Late,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
50,NKI-d23231-00-0095,2011-09-01,NKI-d23231-00-0095_20110901_kalina01_0054.nii.gz,Arterial,Arterial Too Late,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
84,NKI-d23231-00-0038,2016-05-15,NKI-d23231-00-0038_20160515_kalina02_0029.nii.gz,Portal,Portal Too Late,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
119,NKI-d23231-00-0086,2017-08-21,NKI-d23231-00-0086_20170821_kalina03_0004.nii.gz,Portal,Portal Too Early,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
202,NKI-d23231-00-0091,2016-03-14,NKI-d23231-00-0091_20160314_kalina04_0031.nii.gz,Portal,Portal Too Early,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
...,...,...,...,...,...,...,...,...,...,...
7992,NKI-d23231-00-0846,2023-03-10,NKI-d23231-00-0846_20230310_kalina99_5655.nii.gz,Arterial,Arterial Too Late,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
8045,NKI-d23231-00-0868,2022-10-10,NKI-d23231-00-0868_20221010_kalina99_5732.nii.gz,Arterial,Arterial Too Late,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
8057,NKI-d23231-00-0873,2022-08-22,NKI-d23231-00-0873_20220822_kalina99_5745.nii.gz,Arterial,Arterial Too Late,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
8060,NKI-d23231-00-0873,2023-04-21,NKI-d23231-00-0873_20230421_kalina99_5748.nii.gz,Arterial,Arterial Too Late,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,/mnt/rhea/data_private/IRBd23-231/GEPNETs/ARTI...,1,train,1
